In [ ]:
from trustutils import run
run.TRUST_parameters()
run.introduction('MMS Trigonometric 2D', '2026')

run.reset()

NB_PROCS = 2

case = run.addCase('.', 'MMS_Trig2D.data', nbProcs=NB_PROCS)
case.partition()          # trust -partition MMS_Trig2D.data 2 -> PAR_MMS_Trig2D.data + DOM_000*.Zones
run.printCases()
run.runCases()
run.tablePerf()

In [ ]:
from trustutils.files import SonFile
import numpy as np

BUILD = run.BUILD_DIRECTORY

def read_son(case, name):
    """Return (time, values) for probe `name` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{name}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

# The dataset as written is an invariance case, not a decaying mode: the
# uniform initial field b0 = (0, 1) satisfies u.grad(b) = 0 (u = (y, 0) is
# horizontal, By is y-uniform), its Laplacian vanishes, and the Dirichlet
# walls impose exactly (0, 1).  Both components must therefore stay at their
# initial values to machine precision -- the same "no spurious source"
# contract as the Invariance form, here with convection switched on.
t, bx = read_son(case.dataFileName_, 'SONDE_BX')
_, by = read_son(case.dataFileName_, 'SONDE_BY')

err_bx = np.max(np.abs(bx))
err_by = np.max(np.abs(by - 1.0))
print(f"max |Bx|          : {err_bx:.3e}")
print(f"max |By - 1|      : {err_by:.3e}")
print(f"final time        : {t[-1]:.6g}")
assert err_bx < 1e-10, f"Bx not preserved: {err_bx:.3e}"
assert err_by < 1e-10, f"By not preserved: {err_by:.3e}"
print("OK: uniform (0, 1) field is invariant under convection + diffusion.")
